# DFT Geometry Optimization

This notebook runs the fixed-cell ion-position relaxation workflow introduced in Milestone 5. The example is intentionally compact: it uses the local GTH hydrogen path, reuses SCF state between geometry steps, and records the accepted relaxation history.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import GeometryOptimizationConfig, SCFConfig, geometry_demo_system, load_geometry_optimization, optimize_geometry, save_geometry_optimization

The optimizer moves only ion centers. The cell is fixed, and the force source is the current SCF result. A real production DFT relaxer would also need nonlocal forces, better pseudopotentials, spin/k-points where needed, and stress for cell relaxation.

In [ ]:
system = geometry_demo_system("gth-h2", grid_shape=(4, 4, 4))
config = GeometryOptimizationConfig(
    max_steps=2,
    optimizer="lbfgs",
    scf_config=SCFConfig(max_iterations=3, solver="dense", seed=29, convergence_mode="either"),
)
result = optimize_geometry(system, config=config)
summary = result.to_dict()
{
    "status": summary["status"],
    "step_count": summary["step_count"],
    "final_energy": summary["final_energy"],
    "final_max_force": summary["final_max_force"],
    "final_positions": summary["final_positions"],
}

The dense history is scalar-first: energy, force, line-search step size, and SCF status are easy to inspect without loading orbital or density arrays.

In [ ]:
history = summary["history"]
energies = [step["energy"] for step in history]
max_forces = [step["max_force"] for step in history]
steps = [step["index"] for step in history]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
axes[0].plot(steps, energies, marker="o")
axes[0].set_xlabel("geometry step")
axes[0].set_ylabel("total energy / Ha")
axes[0].set_title("accepted energy")
axes[1].plot(steps, max_forces, marker="o", color="tab:red")
axes[1].set_xlabel("geometry step")
axes[1].set_ylabel("max |F| / Ha bohr⁻¹")
axes[1].set_title("force diagnostic");

The NPZ helper stores accepted positions, forces, scalar histories, and JSON metadata. It is a lightweight diagnostics format, not a dense SCF restart file.

In [ ]:
path = Path("/tmp/mlx-atomistic-gth-h2-relaxation.npz")
save_geometry_optimization(path, result, metadata={"system": "gth-h2"})
record = load_geometry_optimization(path)
{
    "saved_steps": int(record.energies.size),
    "positions_shape": record.positions.shape,
    "metadata_system": record.metadata["user"]["system"],
}